# 💬 Módulo 06 — Conversaciones y Sesiones

> **Objetivo:** Mantener el contexto de conversación entre múltiples turnos usando `AgentSession`.

## ¿Por qué sesiones?

Sin sesión, cada llamada a `agent.run()` es **independiente** — el agente no recuerda nada de turnos anteriores.

```
Llamada 1: "Mi color favorito es azul"  →  "Genial!"
Llamada 2: "¿Cuál es mi color favorito?"  →  "No tengo esa información"  ❌
```

Con sesión, el historial se mantiene:

```
Turno 1 (sesión X): "Mi color favorito es azul"  →  "Genial!"
Turno 2 (sesión X): "¿Cuál es mi color favorito?"  →  "Es azul"  ✅
```

## API

```python
session = agent.create_session()           # crear sesión
await agent.run("...", session=session)    # usarla en cada turno
session.state["custom_key"] = "value"      # estado personalizado tipo dict
```


## ⚙️ Setup inicial

Esta celda carga la configuración de Azure OpenAI desde `appsettings.Development.json`
(o variables de entorno) y crea el cliente que reutilizaremos durante todo el notebook.

> 💡 Solo necesitas ejecutar esta celda **una vez** al abrir el notebook.


In [ ]:
# Aseguramos que la raíz del proyecto esté en sys.path para importar `helpers`
import sys
from pathlib import Path

_ROOT = Path.cwd()
# Si abriste el notebook desde la carpeta notebooks/, sube un nivel
if _ROOT.name == "notebooks":
    _ROOT = _ROOT.parent
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from agent_framework import Agent
from helpers.config import create_chat_client
print("✅ Helpers cargados. Endpoint:", create_chat_client().__class__.__name__)


## 1️⃣ Sin sesión (o sesiones distintas) → sin memoria

Demostramos que con sesiones separadas el agente no comparte el historial.


In [ ]:
agent = Agent(
    client=create_chat_client(),
    instructions=(
        "Eres un asistente útil. Responde las preguntas directamente. "
        "Si no sabes algo del contexto, di 'No tengo esa información'."
    ),
)

session_a = agent.create_session()
session_b = agent.create_session()

r1 = await agent.run("Mi color favorito es el azul.", session=session_a)
print(f"Sesión A → {r1.text}\n")

r2 = await agent.run("¿Cuál es mi color favorito?", session=session_b)
print(f"Sesión B → {r2.text}")
print("\n✅ Con sesiones separadas, cada llamada es independiente.")


## 2️⃣ Una sesión → contexto persistente entre turnos

La misma sesión `agent.create_session()` reutilizada en cada llamada conserva el historial completo.


In [ ]:
agent = Agent(
    client=create_chat_client(),
    instructions="Eres un tutor de matemáticas. Recuerda todas las interacciones previas.",
)

session = agent.create_session()

r1 = await agent.run("Mi nombre es Carlos y estoy aprendiendo álgebra.", session=session)
print(f"Turno 1 → {r1.text}\n")

r2 = await agent.run("¿Cuál es mi nombre?", session=session)
print(f"Turno 2 → {r2.text}\n")

r3 = await agent.run("¿Qué materia estoy aprendiendo?", session=session)
print(f"Turno 3 → {r3.text}")
print("\n✅ La sesión mantuvo el contexto entre los 3 turnos.")


## 3️⃣ Estado personalizado en `session.state`

`session.state` es un `dict[str, Any]` donde puedes guardar información de tu aplicación
(user_id, preferencias, contadores, etc.).

Es el equivalente Python del `StateBag` de C#.


In [ ]:
agent = Agent(client=create_chat_client(), instructions="Eres un asistente útil.")
session = agent.create_session()

# Guardar estado personalizado
session.state["user_id"] = "USR-12345"
session.state["session_type"] = "workshop_demo"
session.state["interaction_count"] = 0

await agent.run("¡Hola!", session=session)
session.state["interaction_count"] += 1

print("✅ Estado personalizado:")
for k, v in session.state.items():
    print(f"   {k}: {v}")


## 🎯 Resumen

| Capacidad | Cómo se hace |
|-----------|--------------|
| Crear sesión | `session = agent.create_session()` |
| Usarla en un turno | `await agent.run("...", session=session)` |
| Estado personalizado | `session.state["key"] = value` |

> 💡 **Persistencia:** para guardar conversaciones en disco, base de datos o caché, mira los
> "history providers" (`FileHistoryProvider`, `RedisHistoryProvider`, `CosmosHistoryProvider`).

**Siguiente módulo →** [Módulo 07 — Context Providers](./07_context_providers.ipynb)
